In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
# from tensorflow import keras

# peppers = pd.read_csv("../data/labels.csv")
# df = pd.DataFrame(peppers)


In [7]:
validation_accuracy = np.array([])

tf.random.set_seed(123)

project_dir = Path.cwd()
img_dir = project_dir / "img"

img_height = 180
img_width = 180
batch_size = 32
seed = 123

train_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(180, 180),
    batch_size=32
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    img_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    tf.keras.layers.RandomFlip('horizontal'),
    # the augmentations below all had a negative impact when allowing tf to pick train/validate/test
    # tf.keras.layers.RandomRotation(0.02),
    # tf.keras.layers.RandomZoom(0.05),
    # tf.keras.layers.GaussianNoise(0.1),
    tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)

validation_accuracy = np.append(validation_accuracy, val_acc)

# print("Augmented with random flip: ")
# print(f"\n\nValidation accuracy: {validation_accuracy.mean():.2f}")

model.save('model.h5')


# print("\nValidation accuracy:", val_acc)

Found 197 files belonging to 3 classes.
Using 158 files for training.
Found 197 files belonging to 3 classes.
Using 39 files for validation.


2026-03-28 01:38:40.580870: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-03-28 01:38:40.581042: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-03-28 01:38:40.581055: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-03-28 01:38:40.581212: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-28 01:38:40.581229: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/15


/opt/anaconda3/envs/ml-env/lib/python3.10/site-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
2026-03-28 01:38:41.190447: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 228ms/step - accuracy: 0.2911 - loss: 2.3412 - val_accuracy: 0.5128 - val_loss: 1.0692
Epoch 2/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.3165 - loss: 1.1397 - val_accuracy: 0.2564 - val_loss: 1.1007
Epoch 3/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.3671 - loss: 1.1218 - val_accuracy: 0.2564 - val_loss: 1.2018
Epoch 4/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.3671 - loss: 1.1027 - val_accuracy: 0.2564 - val_loss: 1.1106
Epoch 5/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4747 - loss: 1.0751 - val_accuracy: 0.4615 - val_loss: 1.0765
Epoch 6/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4620 - loss: 1.0448 - val_accuracy: 0.3590 - val_loss: 1.0518
Epoch 7/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.4747 - loss: 1.0230 - val_accuracy: 0.3846 - val_loss: 1.0131
Epoch 8/15
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step - accuracy: 0.5063 - loss: 0.9852 - val_accuracy: 0.7436 - val_loss: 0.8985
Epoch 9/15